# Nornickel — coarse/fine **binary** (~45–60 мин, GPU T4)

Soft binary: **1 logit + BCE** (coarse=1, fine=0). Если в manifest есть coarse+fine одновременно — target **0.5** (редко, низкий вес в sampler).

Тальк — только через сегментацию; кадры с tag_talc и coarse/fine **используются**.

1. Upload `nornickel_coarse_fine_classification.zip` as Dataset
2. Import **этот** notebook (не multilabel!)
3. **GPU T4** → Run All
4. Download `best_coarse_fine_binary.pt`

In [ ]:
!pip -q install opencv-python-headless

In [ ]:
import copy
import csv
import io
import json
import math
import random
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, precision_score, recall_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

IMAGE_SIZE = 384
BATCH_SIZE = 64  # T4: меньше шагов/эпоху; при OOM поставь 32
EPOCHS = 12
PATIENCE = 4
WEIGHT_DECAY = 1e-4
THRESHOLD = 0.5
AMBIG_TARGET = 0.5
AMBIG_SAMPLER_WEIGHT = 0.12
NUM_WORKERS = 0  # Kaggle/Jupyter: workers ломают multiprocessing
PIN_MEMORY = False  # с num_workers=0 только мешает
SEED = 42

EXPERIMENTS = [
    {'name': 'resnet18_cosine', 'model': 'resnet18', 'lr': 3e-4, 'scheduler': 'cosine'},
    {'name': 'resnet34_cosine', 'model': 'resnet34', 'lr': 2e-4, 'scheduler': 'cosine'},
    {'name': 'effnet_b0_cosine', 'model': 'efficientnet_b0', 'lr': 1.8e-4, 'scheduler': 'cosine'},
]

AUG = {
    'crop_scale': (0.72, 1.0), 'crop_ratio': (0.85, 1.18),
    'rotation_deg': 15,
    'hflip_p': 0.5, 'vflip_p': 0.25,
    'brightness': 0.22, 'contrast': 0.26, 'saturation': 0.22, 'hue': 0.025, 'gamma': 0.15,
    'gray_domain_p': 0.30, 'domain_strong_p': 0.18, 'blur_p': 0.12, 'noise_p': 0.18, 'jpeg_p': 0.12, 'cutout_p': 0.10,
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
torch.set_num_threads(4)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print('device:', DEVICE)

In [ ]:
def find_data_root() -> Path:
    for base in [Path('/kaggle/input'), Path('.')]:
        if not base.is_dir():
            continue
        for meta in base.rglob('manifest.csv'):
            root = meta.parent
            if (root / 'images').is_dir():
                return root
    raise FileNotFoundError('manifest.csv + images/ not found — attach nornickel_coarse_fine_classification dataset')


def verify_dataset(root: Path) -> None:
    rows = list(csv.DictReader((root / 'manifest.csv').open(encoding='utf-8-sig')))
    missing = sum(1 for r in rows if not (root / r['rel_path']).is_file())
    both = sum(r['tag_coarse'] == '1' and r['tag_fine'] == '1' for r in rows)
    talc = sum(r['tag_talc'] == '1' for r in rows)
    print(f'dataset OK: rows={len(rows)} missing_images={missing} ambiguous={both} talc_tag={talc}')
    if missing:
        raise FileNotFoundError(f'{missing} images missing under {root}')


DATA_ROOT = find_data_root()
verify_dataset(DATA_ROOT)
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/input').is_dir() else Path('runs_binary')
RUNS_DIR = WORK_DIR / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT:', DATA_ROOT)

In [ ]:
def row_target_and_bucket(row: dict) -> tuple[float, str]:
    coarse = row['tag_coarse'] == '1'
    fine = row['tag_fine'] == '1'
    if coarse and fine:
        return AMBIG_TARGET, 'ambiguous'
    if coarse:
        return 1.0, 'coarse'
    if fine:
        return 0.0, 'fine'
    return 0.5, 'unknown'


def load_manifest(root: Path) -> list[dict]:
    rows = []
    with (root / 'manifest.csv').open(encoding='utf-8-sig', newline='') as f:
        for row in csv.DictReader(f):
            row = dict(row)
            row['image_path'] = str((root / row['rel_path']).resolve())
            target, bucket = row_target_and_bucket(row)
            row['target_coarse'] = target
            row['label_bucket'] = bucket
            rows.append(row)
    return rows


def _gray_domain(img):
    img = TF.adjust_saturation(img, random.uniform(0.04, 0.42))
    img = TF.adjust_brightness(img, random.uniform(0.40, 0.98))
    img = TF.adjust_contrast(img, random.uniform(0.55, 1.18))
    rgb = np.asarray(img).astype(np.float32)
    rgb[:, :, 2] *= random.uniform(1.0, 1.18)
    rgb[:, :, 0] *= random.uniform(0.82, 1.0)
    return Image.fromarray(np.clip(rgb, 0, 255).astype(np.uint8))


def _domain_strong(img):
    rgb = np.asarray(img).astype(np.uint8)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=random.uniform(1.2, 3.0), tileGridSize=(8, 8)).apply(l)
    l = np.clip(l.astype(np.float32) * random.uniform(0.68, 1.22), 0, 255).astype(np.uint8)
    cs = random.uniform(0.04, 0.78)
    a = np.clip(128 + (a.astype(np.float32) - 128) * cs, 0, 255).astype(np.uint8)
    b = np.clip(128 + (b.astype(np.float32) - 128) * cs, 0, 255).astype(np.uint8)
    rgb = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB).astype(np.float32)
    gains = np.array([random.uniform(0.82, 1.12), random.uniform(0.88, 1.12), random.uniform(0.88, 1.20)])
    return Image.fromarray(np.clip(rgb * gains.reshape(1, 1, 3), 0, 255).astype(np.uint8))


def _cutout(img):
    rgb = np.asarray(img).copy()
    h, w = rgb.shape[:2]
    area = h * w * random.uniform(0.02, 0.08)
    ch, cw = int(max(1, area ** 0.5)), int(max(1, area ** 0.5))
    y0, x0 = random.randint(0, max(0, h - ch)), random.randint(0, max(0, w - cw))
    rgb[y0:y0 + ch, x0:x0 + cw] = int(random.uniform(0, 40))
    return Image.fromarray(rgb)


def augment_pil(img: Image.Image) -> Image.Image:
    if random.random() < AUG['hflip_p']:
        img = TF.hflip(img)
    if random.random() < AUG['vflip_p']:
        img = TF.vflip(img)
    if AUG['rotation_deg'] and random.random() < 0.5:
        img = TF.rotate(img, random.uniform(-AUG['rotation_deg'], AUG['rotation_deg']))
    img = TF.adjust_brightness(img, random.uniform(1 - AUG['brightness'], 1 + AUG['brightness']))
    img = TF.adjust_contrast(img, random.uniform(1 - AUG['contrast'], 1 + AUG['contrast']))
    img = TF.adjust_saturation(img, random.uniform(1 - AUG['saturation'], 1 + AUG['saturation']))
    img = TF.adjust_hue(img, random.uniform(-AUG['hue'], AUG['hue']))
    img = TF.adjust_gamma(img, random.uniform(1 - AUG['gamma'], 1 + AUG['gamma']))
    if random.random() < AUG['gray_domain_p']:
        img = _gray_domain(img)
    if random.random() < AUG['domain_strong_p']:
        img = _domain_strong(img)
    if random.random() < AUG['blur_p']:
        k = random.choice([3, 5])
        img = Image.fromarray(cv2.GaussianBlur(np.asarray(img), (k, k), random.uniform(0.2, 1.1)))
    if random.random() < AUG['noise_p']:
        rgb = np.asarray(img).astype(np.float32) + np.random.normal(0, random.uniform(1, 7), np.asarray(img).shape)
        img = Image.fromarray(np.clip(rgb, 0, 255).astype(np.uint8))
    if random.random() < AUG['jpeg_p']:
        buf = io.BytesIO(); img.save(buf, format='JPEG', quality=random.randint(58, 94)); buf.seek(0)
        img = Image.open(buf).convert('RGB')
    if random.random() < AUG['cutout_p']:
        img = _cutout(img)
    return img


class OreBinaryDS(Dataset):
    def __init__(self, rows, train: bool):
        self.rows = rows
        self.train = train
        self.rrc = T.RandomResizedCrop((IMAGE_SIZE, IMAGE_SIZE), scale=AUG['crop_scale'], ratio=AUG['crop_ratio'])
        self.norm = T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        with Image.open(row['image_path']) as img:
            img = ImageOps.exif_transpose(img).convert('RGB')
            if self.train:
                img = self.rrc(img)
                img = augment_pil(img)
            else:
                img = T.Resize((IMAGE_SIZE, IMAGE_SIZE))(img)
            x = self.norm(T.ToTensor()(img))
        y = torch.tensor(float(row['target_coarse']), dtype=torch.float32)
        return x, y, row['label_bucket']

    def sampler_weights(self):
        counts = {}
        for r in self.rows:
            b = r['label_bucket']
            if b == 'ambiguous':
                continue
            counts[b] = counts.get(b, 0) + 1
        max_c = max(counts.values()) if counts else 1
        out = []
        for r in self.rows:
            b = r['label_bucket']
            if b == 'ambiguous':
                out.append(AMBIG_SAMPLER_WEIGHT)
            else:
                out.append(max_c / max(counts.get(b, 1), 1))
        return out


ALL_ROWS = load_manifest(DATA_ROOT)
TRAIN_ROWS = [r for r in ALL_ROWS if r['subset'] == 'train']
VAL_ROWS = [r for r in ALL_ROWS if r['subset'] == 'val']
amb_train = sum(r['label_bucket'] == 'ambiguous' for r in TRAIN_ROWS)
print(f'train={len(TRAIN_ROWS)} val={len(VAL_ROWS)} ambiguous_train={amb_train}')

In [ ]:
def make_model(name: str) -> nn.Module:
    m = getattr(models, name)(weights='DEFAULT')
    if hasattr(m, 'fc'):
        m.fc = nn.Linear(m.fc.in_features, 1)
    elif hasattr(m, 'classifier'):
        if isinstance(m.classifier, nn.Sequential):
            m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 1)
        else:
            m.classifier = nn.Linear(m.classifier.in_features, 1)
    return m


def compute_metrics(y_true, y_prob, buckets):
    yt = np.array(y_true, dtype=np.float32)
    yp = np.array(y_prob, dtype=np.float32)
    bk = np.array(buckets)
    clean = np.isin(bk, ['coarse', 'fine'])
    amb = bk == 'ambiguous'
    metrics = {}
    if clean.any():
        ct, cp = yt[clean], yp[clean]
        pred = (cp >= THRESHOLD).astype(np.int32)
        bin_true = ct.astype(np.int32)
        metrics['clean_f1'] = float(f1_score(bin_true, pred, zero_division=0))
        metrics['clean_accuracy'] = float(accuracy_score(bin_true, pred))
        metrics['clean_auc'] = float(roc_auc_score(bin_true, cp)) if len(np.unique(bin_true)) > 1 else float('nan')
    else:
        metrics.update({'clean_f1': 0.0, 'clean_accuracy': 0.0, 'clean_auc': float('nan')})
    if amb.any():
        ap = yp[amb]
        metrics['ambiguous_mae'] = float(mean_absolute_error(np.full_like(ap, AMBIG_TARGET), ap))
    else:
        metrics['ambiguous_mae'] = float('nan')
    amb_mae = metrics['ambiguous_mae']
    if not math.isnan(amb_mae):
        metrics['composite_score'] = 0.85 * metrics['clean_f1'] + 0.15 * (1.0 - min(amb_mae, 0.5) / 0.5)
    else:
        metrics['composite_score'] = metrics['clean_f1']
    return metrics


def make_scheduler(name: str, opt, loader_len: int, max_lr: float):
    if name == 'cosine':
        return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
    if name == 'onecycle':
        return torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lr, epochs=EPOCHS, steps_per_epoch=max(loader_len, 1))
    raise ValueError(name)


def train_variant(spec: dict) -> dict:
    run_dir = RUNS_DIR / spec['name']
    run_dir.mkdir(parents=True, exist_ok=True)
    train_ds = OreBinaryDS(TRAIN_ROWS, train=True)
    val_ds = OreBinaryDS(VAL_ROWS, train=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=WeightedRandomSampler(train_ds.sampler_weights(), len(train_ds), True), num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model = make_model(spec['model']).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=spec['lr'], weight_decay=WEIGHT_DECAY)
    sched = make_scheduler(spec['scheduler'], opt, len(train_loader), spec['lr'])
    crit = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    step_per_batch = spec['scheduler'] == 'onecycle'

    best_score, stale, best_state, best_metrics = -1.0, 0, None, {}
    t0 = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for x, y, _ in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                loss = crit(model(x).squeeze(-1), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if step_per_batch:
                sched.step()
        if not step_per_batch:
            sched.step()

        model.eval()
        y_true, y_prob, buckets = [], [], []
        with torch.no_grad():
            for x, y, b in val_loader:
                p = torch.sigmoid(model(x.to(DEVICE)).squeeze(-1)).cpu().numpy()
                y_true.extend(y.numpy().tolist())
                y_prob.extend(p.tolist())
                buckets.extend(b)
        vm = compute_metrics(y_true, y_prob, buckets)
        score = vm['composite_score']
        print(f"  {spec['name']} epoch {epoch}/{EPOCHS}: composite={score:.4f} clean_f1={vm.get('clean_f1', 0):.4f}")
        if score > best_score:
            best_score, stale = score, 0
            best_state = copy.deepcopy(model.state_dict())
            best_metrics = vm
        else:
            stale += 1
        if stale >= PATIENCE:
            break

    if best_state:
        torch.save({'model_state_dict': best_state, 'task': 'coarse_fine_binary', 'metrics': best_metrics, 'spec': spec}, run_dir / 'best.pt')
    return {'run': spec['name'], 'best_composite': best_score, 'clean_f1': best_metrics.get('clean_f1', 0), 'minutes': round((time.time() - t0) / 60, 1)}

In [ ]:
results = []
for spec in EXPERIMENTS:
    print('\n' + '=' * 50, spec['name'], '=' * 50)
    results.append(train_variant(spec))

ranked = sorted(results, key=lambda r: r['best_composite'], reverse=True)
(WORK_DIR / 'grid_summary_binary.json').write_text(json.dumps({'results': ranked}, indent=2), encoding='utf-8')
print('\nBINARY GRID:')
for r in ranked:
    print(f"  {r['run']}: composite={r['best_composite']:.4f} clean_f1={r['clean_f1']:.4f} ({r['minutes']} min)")

best = ranked[0]
src = RUNS_DIR / best['run'] / 'best.pt'
dst = WORK_DIR / 'best_coarse_fine_binary.pt'
shutil.copy2(src, dst)
print(f"\nBest binary: {best['run']} -> {dst}")